# Membership Query Synthesis in Active Learning - Starter Notebook
This notebook synthesizes query points from feature ranges and labels them with an oracle function to simulate membership query synthesis.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [ ]:
rng = np.random.default_rng(42)

def oracle_label(X):
    return ((X[:, 0] ** 2 + 0.6 * X[:, 1]) > 0.2).astype(int)

X_test = rng.uniform(-2, 2, size=(400, 2))
y_test = oracle_label(X_test)

X_labeled = rng.uniform(-2, 2, size=(20, 2))
y_labeled = oracle_label(X_labeled)

In [ ]:
records = []
for round_id in range(8):
    model = LogisticRegression(max_iter=1000)
    model.fit(X_labeled, y_labeled)
    acc = accuracy_score(y_test, model.predict(X_test))
    records.append({'round': round_id, 'labeled_count': len(X_labeled), 'test_accuracy': acc})

    synthesized_candidates = rng.uniform(-2, 2, size=(400, 2))
    probs = model.predict_proba(synthesized_candidates)[:, 1]
    uncertainty = np.abs(probs - 0.5)
    query_batch = synthesized_candidates[np.argsort(uncertainty)[:25]]
    query_labels = oracle_label(query_batch)

    X_labeled = np.vstack([X_labeled, query_batch])
    y_labeled = np.concatenate([y_labeled, query_labels])

pd.DataFrame(records).round(3)